# py-googletraffic on Google Colab 🚦

This notebook demonstrates how to use **py-googletraffic** in Google Colab to analyze real-time traffic data.

**What you'll learn:**
- Install py-googletraffic in Google Colab
- Capture traffic data from Google Maps
- Visualize traffic patterns
- Save results to Google Drive

**Time required:** ~10 minutes

---

## 📋 Before You Start

You'll need:
1. **Google Maps API Key** with Maps JavaScript API enabled
   - Get one at: https://developers.google.com/maps/get-started
2. **Google account** (for Colab)

---

## Step 1: Installation Setup (Run Once)

This cell installs all required dependencies. **Run this once** when you first open the notebook.

⏱️ Takes ~2-3 minutes on first run

In [ ]:
%%bash
# Install Chrome and ChromeDriver
echo "📦 Installing Chrome and ChromeDriver..."
apt-get update -qq
apt-get install -y chromium-chromedriver chromium-browser >/dev/null 2>&1

# Install GDAL (required for geospatial operations)
echo "📦 Installing GDAL..."
apt-get install -y gdal-bin libgdal-dev >/dev/null 2>&1

echo "✅ System dependencies installed!"

In [ ]:
# Set environment variables
import os
os.environ['GDAL_CONFIG'] = '/usr/bin/gdal-config'

# Install py-googletraffic
print("📦 Installing py-googletraffic...")
!pip install -q py-googletraffic google-colab-selenium

# Verify installation
import googletraffic as gt
print(f"✅ py-googletraffic version: {gt.__version__}")
print("✅ Setup complete! Ready to use.")

## Step 2: Configuration

Enter your Google Maps API key below. For security, use Colab's Secrets feature:
1. Click the 🔑 key icon in the left sidebar
2. Add a new secret named `GOOGLE_MAPS_API_KEY`
3. Paste your API key
4. Toggle "Notebook access" on

Alternatively, paste your key directly in the code below (not recommended for shared notebooks).

In [ ]:
# Option 1: Use Colab Secrets (recommended)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except:
    # Option 2: Enter directly (not recommended for shared notebooks)
    from getpass import getpass
    GOOGLE_API_KEY = getpass('Enter your Google Maps API key: ')
    print("✅ API key entered")

# Verify key is set
if GOOGLE_API_KEY:
    print(f"✅ API key configured (length: {len(GOOGLE_API_KEY)} characters)")
else:
    print("❌ API key not set. Please configure it above.")

## Step 3: Import Libraries

In [ ]:
import googletraffic as gt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from datetime import datetime

print("✅ Libraries imported")

## Example 1: Simple Traffic Capture

Let's capture traffic data around Times Square, New York City.

In [ ]:
# Define location (Times Square, NYC)
location = (40.7580, -73.9855)  # (latitude, longitude)

# Capture traffic data
print("🚦 Capturing traffic data...")
traffic_raster = gt.make_raster(
    location=location,
    height=800,
    width=800,
    zoom=14,
    google_key=GOOGLE_API_KEY,
    wait_time=3
)

print(f"✅ Captured raster shape: {traffic_raster.shape}")
print(f"✅ Traffic levels present: {np.unique(traffic_raster)}")

### Visualize Traffic Heatmap

In [ ]:
# Create custom colormap matching Google Maps
colors = ['white', 'green', 'orange', 'red', 'darkred']
cmap = ListedColormap(colors)

# Plot traffic heatmap
plt.figure(figsize=(12, 10))
plt.imshow(traffic_raster, cmap=cmap, interpolation='nearest', vmin=0, vmax=4)
plt.colorbar(label='Traffic Level (0=no data, 1=green, 2=orange, 3=red, 4=dark red)', 
             ticks=[0, 1, 2, 3, 4])
plt.title(f'Real-Time Traffic - Times Square, NYC\n{datetime.now().strftime("%Y-%m-%d %H:%M UTC")}',
          fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

# Print statistics
total_pixels = traffic_raster.size
traffic_pixels = (traffic_raster > 0).sum()
print(f"\n📊 Statistics:")
print(f"   Coverage: {100 * traffic_pixels / total_pixels:.1f}% of area has traffic info")
if traffic_pixels > 0:
    print(f"   Average traffic level: {traffic_raster[traffic_raster > 0].mean():.2f}")
    for level, name in enumerate(['No data', 'Green', 'Orange', 'Red', 'Dark Red']):
        count = (traffic_raster == level).sum()
        if count > 0:
            print(f"   {name}: {count:,} pixels")

## Example 2: Compare Multiple Locations

Compare traffic at different locations in New York City.

In [ ]:
import time

# Define locations to compare
locations = {
    'Times Square': (40.7580, -73.9855),
    'Central Park': (40.7829, -73.9654),
    'Brooklyn Bridge': (40.7061, -73.9969),
    'JFK Airport': (40.6413, -73.7781)
}

results = {}
rasters = {}

for name, coords in locations.items():
    print(f"🚦 Capturing {name}...")
    
    raster = gt.make_raster(
        location=coords,
        height=400,
        width=400,
        zoom=14,
        google_key=GOOGLE_API_KEY
    )
    
    rasters[name] = raster
    
    # Calculate average traffic level (excluding no-data pixels)
    traffic_pixels = raster[raster > 0]
    avg_traffic = traffic_pixels.mean() if len(traffic_pixels) > 0 else 0
    results[name] = avg_traffic
    
    print(f"   Average traffic: {avg_traffic:.2f}")
    
    # Be respectful to Google's servers
    time.sleep(2)

print("\n✅ All locations captured!")

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(f'NYC Traffic Comparison - {datetime.now().strftime("%Y-%m-%d %H:%M")}',
             fontsize=16, fontweight='bold')

# Plot each location's traffic map
for idx, (name, raster) in enumerate(rasters.items()):
    if idx < 4:
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        im = ax.imshow(raster, cmap=cmap, interpolation='nearest', vmin=0, vmax=4)
        ax.set_title(f'{name}\nAvg: {results[name]:.2f}', fontweight='bold')
        ax.axis('off')

# Traffic level comparison bar chart
ax_bar = axes[1, 2]
bar_colors = ['green' if v < 1.5 else 'orange' if v < 2.5 else 'red' for v in results.values()]
ax_bar.barh(list(results.keys()), list(results.values()), color=bar_colors)
ax_bar.set_xlabel('Average Traffic Level')
ax_bar.set_title('Traffic Comparison', fontweight='bold')
ax_bar.set_xlim(0, 4)
ax_bar.grid(axis='x', alpha=0.3)

# Add colorbar
cbar_ax = axes[0, 2]
cbar_ax.axis('off')
cbar = fig.colorbar(im, ax=cbar_ax, orientation='vertical', fraction=0.8)
cbar.set_label('Traffic Level', fontsize=12)
cbar.set_ticks([0, 1, 2, 3, 4])
cbar.set_ticklabels(['No data', 'Green', 'Orange', 'Red', 'Dark Red'])

plt.tight_layout()
plt.show()

## Example 3: Bounding Box Analysis

Capture traffic data for an entire area using a bounding box (e.g., Manhattan).

In [ ]:
# Define bounding box around Manhattan
manhattan_bbox = {
    'xmin': -74.0479,  # West
    'xmax': -73.9067,  # East
    'ymin': 40.6829,   # South
    'ymax': 40.8820    # North
}

print("🚦 Capturing Manhattan traffic (this may take 30-60 seconds)...")
manhattan_traffic = gt.make_raster_from_bbox(
    bbox=manhattan_bbox,
    height_px=1000,
    google_key=GOOGLE_API_KEY,
    zoom=12
)

print(f"✅ Manhattan raster shape: {manhattan_traffic.shape}")

# Visualize
plt.figure(figsize=(10, 14))
plt.imshow(manhattan_traffic, cmap=cmap, interpolation='nearest', vmin=0, vmax=4)
plt.colorbar(label='Traffic Level', ticks=[0, 1, 2, 3, 4])
plt.title(f'Manhattan Traffic Heatmap\n{datetime.now().strftime("%Y-%m-%d %H:%M")}',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Example 4: Save to Google Drive

Save traffic rasters as GeoTIFF files to your Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
output_dir = '/content/drive/MyDrive/traffic_data'
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Output directory: {output_dir}")

In [ ]:
# Save as GeoTIFF
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
output_path = f"{output_dir}/times_square_traffic_{timestamp}.tif"

print(f"💾 Saving GeoTIFF to: {output_path}")
saved_path = gt.make_raster(
    location=(40.7580, -73.9855),
    height=1000,
    width=1000,
    zoom=14,
    google_key=GOOGLE_API_KEY,
    output_path=output_path
)

print(f"✅ Saved to: {saved_path}")
print(f"📂 File size: {os.path.getsize(saved_path) / 1024:.1f} KB")
print("\n💡 You can now download this file from Google Drive or open it in QGIS/ArcGIS!")

## Next Steps

🎉 **Congratulations!** You've successfully used py-googletraffic on Google Colab.

### Try these next:

1. **Time series analysis:** Capture the same location at different times
2. **Custom locations:** Try your own city or neighborhood
3. **Polygon analysis:** Use custom polygons for specific areas
4. **Export data:** Save results as CSV for further analysis

### Resources:

- 📖 [Full Documentation](https://github.com/kwahalf/py-googletraffic)
- 📓 [Colab Setup Guide](https://github.com/kwahalf/py-googletraffic/blob/main/COLAB.md)
- 🚀 [Quickstart Guide](https://github.com/kwahalf/py-googletraffic/blob/main/QUICKSTART.md)
- 💡 [More Examples](https://github.com/kwahalf/py-googletraffic/tree/main/examples)

### Get Help:

- 🐛 [Report Issues](https://github.com/kwahalf/py-googletraffic/issues)
- 💬 [Ask Questions](https://github.com/kwahalf/py-googletraffic/discussions)

---

**Happy mapping! 🗺️🚦**